# s4 — PCN-style partial + complete pair generation

**This notebook turns each model into a training pair: a complete colored ground truth (GT) and
PCN-style partial scans.** It is a superset of `s3`:

- **PART 1 (cells below, = the s3 pipeline):** extract meshes, EPFL-clean, sample the **dense colored
  cloud** (~1M xyzrgb pts), and FPS it to the **complete GT**. Runs once per Colab session.
- **PART 2 (new, at the bottom):** generate **PCN-style partials** from the mesh + dense cloud, by
  **three** methods, and hand off the pairs.

The research question is *"does color help completion, and when?"* — so Part 2 is deliberately an
**exploration**: we build three partial methods, **report the tradeoff of each in its own cell**, visualize
them side by side, and let the supervisor/teammate compare. Decisions are recorded in `docs/adr/0005-*`;
terms in `CONTEXT.md`.


In [ ]:

SHARED_ROOT     = "/content/drive/MyDrive/Point Cloud Completion"
ORTAK           = f"{SHARED_ROOT}/Ortak"                  # the common folder
SHARED_DATA_DIR = f"{ORTAK}/data/shapenetcore"           # the ShapeNet zips live here


RUN_TAG         = "s4"
OUT_DIR         = f"{ORTAK}/data_{RUN_TAG}"        # outputs land here


WORK_DIR = "/content/work"
RAW_DIR  = f"{WORK_DIR}/raw"                        # models extracted here from the shared zips
PLY_DIR  = f"{WORK_DIR}/colored_pc"                # PLYs written locally first, copied to OUT_DIR at end


CATEGORIES = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
SYNSETS = list(CATEGORIES.values())


N_PER_CAT = 20          # models per category
N_POINTS  = 1_048_576   # points CloudCompare samples per model
SEED      = 42          # fixes WHICH models get picked -> same selection for both teammates

HF_REPO_ID = "ShapeNet/ShapeNetCore"   # only used by the OPTIONAL fallback download cell

# --- EPFL clean-mesh path (OPTIONAL second pipeline; speckle path above stays as-is) ---
# The EPFL method removes interior duplicate faces BEFORE sampling, killing the
# salt-and-pepper color speckle. It needs pymeshlab (installed in Cell B).
EPFL_DIR        = f"{WORK_DIR}/colored_pc_epfl"   # clean-mesh PLYs land here (local)
EPFL_OUT_DIR    = f"{ORTAK}/colored_pc_epfl_{RUN_TAG}"  # shared copy of the EPFL outputs
OCCLUSION_THRESHOLD = 0.0   # faces with occlusion < this are also dropped (0 = keep all visible)

# --- Post-processing knobs (downsample dense 2^20 EPFL clouds -> target GT) ---
N_GT_TARGET = 8192    # points per GT cloud after downsampling (8192 or 16384)
GT_SEED     = 123     # fixes which points are kept -> reproducible GT
COLOR_FRAC  = 0.0     # 0.0 = pure geometric FPS. >0 = that fraction of the budget
                      # is pulled toward color edges (geometry base stays intact).
                      # START AT 0.0, look at the result, raise only if edges are lost.
KNN_COLOR   = 16      # neighbors for the per-point color-gradient estimate
GT_DIR      = f"{WORK_DIR}/gt"            # GT previews land here (local)
GT_OUT_DIR  = f"{ORTAK}/gt_{RUN_TAG}"     # shared copy of the GT


In [ ]:
# ============================================================
# Cell A — Configuration  (RUN ME FIRST)
# ------------------------------------------------------------
from google.colab import drive
import os

drive.mount('/content/drive')                      # mounts each user's My Drive (+ their shortcuts)

# --- Setup local dirs, and fail LOUDLY if the shared shortcut isn't in place ---
for d in (WORK_DIR, RAW_DIR, PLY_DIR, EPFL_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.isdir(ORTAK), (
    f"\n[STOP] Can't see the shared folder:\n  {ORTAK}\n"
    "Add the 'Point Cloud Completion' shortcut to your My Drive:\n"
    "Drive (web) -> Shared with me -> right-click -> Organize -> Add shortcut to Drive -> My Drive."
)
os.makedirs(OUT_DIR, exist_ok=True)
print("OK. Reading data from:", SHARED_DATA_DIR)
print("    Writing outputs to:", OUT_DIR)


In [ ]:
# ============================================================
# Cell B — Install CloudCompare + a virtual display, and Python deps
# ------------------------------------------------------------
# CloudCompare is a DESKTOP application (not a pip package): we install the binary
# onto the Colab VM with apt. It needs a display even in command-line mode, so we
# also install xvfb (X Virtual FrameBuffer) and will launch it via `xvfb-run`.
# This is the most fragile cell; the REAL functional test is Cell F (sampling one mesh).
# ============================================================
import shutil, subprocess

# System packages (Colab runs Ubuntu 22.04, whose 'universe' repo ships cloudcompare).
subprocess.run("apt-get update -qq", shell=True, check=True)
subprocess.run("apt-get install -y -qq cloudcompare xvfb", shell=True, check=True)

# Python deps not preinstalled on Colab (open3d reads/checks PLY colors; pymeshlab
# runs the EPFL ambient-occlusion step used by the optional clean-mesh path).
subprocess.run("pip install -q -U huggingface_hub open3d pymeshlab", shell=True, check=True)

# Locate the binary (apt installs it as 'CloudCompare', capital C).
CC_BIN = shutil.which("CloudCompare") or shutil.which("cloudcompare")
assert CC_BIN, ("CloudCompare not found after apt install. If this Colab image lacks the "
                "package, we switch to a flatpak/AppImage install in this cell.")
print("CloudCompare:", CC_BIN, "| xvfb-run present:", bool(shutil.which("xvfb-run")))
print("Deps ready. Functional test is Cell F (sampling one mesh).")


In [ ]:
# ============================================================
# Cell C — Use the shared data in Ortak (NO download)
# ------------------------------------------------------------
import os

missing = [s for s in SYNSETS if not os.path.exists(f"{SHARED_DATA_DIR}/{s}.zip")]
assert not missing, (
    f"\n[STOP] These zips are missing from {SHARED_DATA_DIR}:\n  "
    + ", ".join(f"{s}.zip" for s in missing)
    + "\nCheck the shared folder and your Drive shortcut, or run the OPTIONAL "
      "HF-download cell below to fetch them."
)
for s in SYNSETS:
    p = f"{SHARED_DATA_DIR}/{s}.zip"
    print(f"  {s}.zip: OK ({os.path.getsize(p) / 1e6:.0f} MB)")
print("All 3 category zips present in the shared folder — no download needed.")


In [ ]:
# ============================================================
# Cell C-ALT — OPTIONAL fallback: download the 3 zips from HuggingFace
# ------------------------------------------------------------
# DO NOT run this normally. It exists only to (re)populate Ortak/data/shapenetcore/

"""
import os
from huggingface_hub import snapshot_download
from google.colab import userdata

os.makedirs(SHARED_DATA_DIR, exist_ok=True)
snapshot_download(
    repo_id=HF_REPO_ID, repo_type="dataset",
    local_dir=SHARED_DATA_DIR,
    allow_patterns=[f"{s}.zip" for s in SYNSETS],     # only the 3 categories we use
    token=userdata.get("HF_TOKEN"),
)
print("Downloaded the 3 category zips into:", SHARED_DATA_DIR)

"""

In [ ]:
# ============================================================
# Cell D — Inspect the on-disk layout BEFORE writing sampling code
# ------------------------------------------------------------
# We assume nothing about v1 vs v2. Peek inside one zip, extract ONE model, print
# its folder tree, and show the .mtl so we can SEE how textures are referenced
# (the OBJ -> MTL -> image "color chain"). If color goes missing later, it is
# because a link in this chain broke.
# ============================================================
import os, zipfile

probe_synset = SYNSETS[0]
probe_zip = f"{SHARED_DATA_DIR}/{probe_synset}.zip"      # read directly from the shared folder

with zipfile.ZipFile(probe_zip) as zf:
    names = zf.namelist()
    print(f"{probe_synset}.zip — {len(names)} entries. First 15:")
    for n in names[:15]:
        print("   ", n)

    obj_entries = [n for n in names if n.endswith("models/model_normalized.obj")]
    print(f"\nmodel_normalized.obj entries: {len(obj_entries)}")
    one_obj = obj_entries[0]
    model_dir = one_obj[: one_obj.index("models/")]      # "<synset>/<model_id>/"
    members = [n for n in names if n.startswith(model_dir)]
    zf.extractall(path=f"{WORK_DIR}/_probe", members=members)

probe_root = os.path.join(f"{WORK_DIR}/_probe", model_dir)
print(f"\nExtracted one model to: {probe_root}\nTree:")
for root, dirs, files in os.walk(probe_root):
    lvl = root.replace(probe_root, "").count(os.sep)
    print("   " * lvl, os.path.basename(root.rstrip("/")) + "/")
    for f in files:
        print("   " * (lvl + 1), f)

mtl = os.path.join(probe_root, "models", "model_normalized.mtl")
print("\n--- model_normalized.mtl (first 25 lines) ---")
if os.path.exists(mtl):
    with open(mtl) as fh:
        for i, line in zip(range(25), fh):
            print("   ", line.rstrip())
else:
    print("   (no .mtl at expected path — check the tree above)")


In [ ]:
# ============================================================
# Cell E — Extract N_PER_CAT models per category (DETERMINISTICALLY)
# ------------------------------------------------------------
# Each model's WHOLE folder (models/ + images/) is extracted intact, so the OBJ's
# relative texture paths resolve and CloudCompare can read color (see Cell D).
# ============================================================
import os, zipfile, random

selected = {}    # synset -> list of (model_id, model_dir_abspath)

for synset in SYNSETS:
    with zipfile.ZipFile(f"{SHARED_DATA_DIR}/{synset}.zip") as zf:   # read from shared folder
        names = zf.namelist()
        # model_id -> its model directory prefix ("<synset>/<model_id>/")
        dir_by_id = {}
        for n in names:
            if n.endswith("models/model_normalized.obj"):
                d = n[: n.index("models/")]
                dir_by_id[d.rstrip("/").split("/")[-1]] = d

        # Reproducible RANDOM selection: sorted() fixes the candidate order across
        # machines, random.Random(SEED) fixes which N_PER_CAT we draw from it.
        rng_sel = random.Random(SEED)
        chosen = rng_sel.sample(sorted(dir_by_id), min(N_PER_CAT, len(dir_by_id)))
        print(f"{synset}: {len(dir_by_id)} models -> extracting {len(chosen)}")

        roots = []
        for mid in chosen:
            d = dir_by_id[mid]
            zf.extractall(path=RAW_DIR, members=[n for n in names if n.startswith(d)])
            roots.append((mid, os.path.join(RAW_DIR, d)))
        selected[synset] = roots

print("\nExtracted per category:", {s: len(v) for s, v in selected.items()})


In [ ]:
# ============================================================
# Cell F — The CloudCompare sampling step (+ functional test on ONE model)
# ------------------------------------------------------------
# We keep EPFL's exact command and run it headless via xvfb. Flags:
#   -SILENT              no GUI / no prompts
#   -AUTO_SAVE OFF       don't save after every command
#   -C_EXPORT_FMT PLY    export clouds as PLY (stores per-point r,g,b)
#   -O <obj>             open the mesh
#   -SAMPLE_MESH POINTS N    scatter N color+geometry points over the surface
#   -SAVE_CLOUDS         write the result
# CloudCompare saves "<obj>_SAMPLED_POINTS.ply" NEXT TO the input; we move/rename it.
# ============================================================
import os, glob, shutil, subprocess
import numpy as np, open3d as o3d

def sample_colored_cloud(obj_path, out_ply, n_points=N_POINTS, cc_bin=CC_BIN):
    """Naive colored sampling at N_POINTS (raw, dense; no dual-face fix -> speckle expected)."""
    cmd = (f'xvfb-run -a {cc_bin} -SILENT -AUTO_SAVE OFF -C_EXPORT_FMT PLY '
           f'-O "{obj_path}" -SAMPLE_MESH POINTS {n_points} -SAVE_CLOUDS')
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    produced = [p for p in glob.glob(os.path.join(os.path.dirname(obj_path), "*.ply"))
                if "SAMPLED_POINTS" in os.path.basename(p)]
    if not produced:
        raise RuntimeError(f"No sampled PLY produced for {obj_path}")
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    shutil.move(produced[0], out_ply)
    return out_ply

# --- functional smoke test on one model (this exercises the Cell B install end-to-end) ---
ts = SYNSETS[0]
tid, troot = selected[ts][0]
tobj = os.path.join(troot, "models", "model_normalized.obj")
tout = sample_colored_cloud(tobj, f"{PLY_DIR}/{ts}/{tid}.ply")

pc = o3d.io.read_point_cloud(tout)
cols = np.asarray(pc.colors)
print(f"OK -> {tout}  ({os.path.getsize(tout) / 1e6:.1f} MB)")
print(f"points={len(pc.points)}  has_colors={pc.has_colors()}  color_std={cols.std():.4f}")
print("Color present." if pc.has_colors() and cols.std() > 0.01
      else "WARNING: near-constant/no color (textureless model or broken texture chain).")


In [ ]:
# ============================================================
# Cell F-EPFL — Optional EPFL path: remove interior duplicate faces, THEN sample
# ------------------------------------------------------------
# This is the EPFL/MMSPG method (arXiv:2201.06935), ported and kept as a SEPARATE
# path so you can compare against the speckle output from Cell F/G.
#   1) pymeshlab ambient_occlusion -> per-face "how visible from outside" quality
#   2) among faces sharing the SAME 3 vertices, keep only the highest-occlusion one
#      (the exterior-facing face); drop the rest -> kills dual-face color speckle
#   3) rewrite a clean .obj (+ copy its .mtl and textures so color survives)
#   4) sample the CLEAN mesh with CloudCompare -> colored PLY
# NOTE: we intentionally DROP EPFL's final voxelization step (their step 5). Your
# pipeline keeps raw float points; voxelizing would defeat that. Output is a raw
# colored PLY, directly comparable to the speckle PLYs.
# ============================================================
import os, glob, shutil, subprocess
import numpy as np, open3d as o3d

# pymeshlab's ambient_occlusion is memory/state-heavy, so EPFL runs it in a
# separate process. We replicate that: a tiny helper script invoked via subprocess.
_AO_HELPER = "/content/_epfl_ao.py"
with open(_AO_HELPER, "w") as fh:
    fh.write(
        "import sys, numpy as np, pymeshlab\n"
        "in_dir, in_file, fm_file, fq_file = sys.argv[1:5]\n"
        "import os; os.chdir(in_dir)\n"
        "ms = pymeshlab.MeshSet(); ms.load_new_mesh(in_file)\n"
        "ms.compute_scalar_ambient_occlusion()\n"   # pymeshlab >=2022 API (was ambient_occlusion)
        "m = ms.mesh(0)\n"
        "np.save(fm_file, m.face_matrix())\n"
        "np.save(fq_file, m.face_scalar_array())\n"   # pymeshlab >=2022 (was face_quality_array)
    )

def _duplicate_faces_to_delete(face_matrix, face_quality, threshold=0.0):
    """EPFL logic, VECTORIZED for speed (was O(faces^2) -> now O(faces log faces),
    ~400x faster on large meshes). Faces sharing the SAME 3 vertices (any permutation)
    get the same sorted-vertex key; within each group we keep the highest-occlusion
    face and drop the rest. Also drop faces with occlusion < threshold.
    NOTE: on rare DEGENERATE faces (repeated vertices) with TIED occlusion values this
    can pick a different tied face than the original loop -- a few faces out of millions,
    visually irrelevant. Flagged to revisit later if exact parity is ever needed."""
    keys = np.sort(face_matrix, axis=1)                       # canonical per-face key
    order = np.lexsort((keys[:, 2], keys[:, 1], keys[:, 0]))  # group identical keys together
    ks = keys[order]
    change = np.any(ks[1:] != ks[:-1], axis=1)
    grp_start = np.concatenate(([0], np.nonzero(change)[0] + 1))
    grp_end = np.concatenate((grp_start[1:], [len(order)]))

    to_delete = np.zeros(face_matrix.shape[0], dtype=bool)
    for s, e in zip(grp_start, grp_end):
        members = order[s:e]
        if len(members) == 1:
            continue
        members = np.sort(members)                            # original index order
        q = face_quality[members]
        keep = members[np.argmax(q)]                          # first face attaining group max
        to_delete[members[members != keep]] = True
    to_delete = to_delete | (face_quality < threshold)
    return to_delete

def _ensure_face_correspondence(obj_lines, face_matrix):
    """pymeshlab may silently skip some faces on load; align the raw .obj face list
    with pymeshlab's face_matrix so deletion indices line up. Returns None if
    they can't be reconciled (then we skip this model)."""
    nonempty = [(i, x) for i, x in enumerate(obj_lines) if len(x) > 0]
    fobj = [[x.split()[1].split('/')[0], x.split()[2].split('/')[0],
             x.split()[3].split('/')[0]] for _, x in nonempty if x[0] == 'f']
    fobj = np.asarray(fobj, dtype=int) - 1
    if np.array_equal(face_matrix, fobj):
        return obj_lines
    fidx = [i for i, x in nonempty if x[0] == 'f']
    dropped = []
    while fobj.shape[0] > face_matrix.shape[0]:
        corr = (face_matrix == fobj[:face_matrix.shape[0], :]).all(axis=1)
        if not corr.all():
            d = np.nonzero(np.logical_not(corr))[0][0]
            fobj = np.delete(fobj, d, axis=0)
            dropped.append(fidx.pop(d))
        else:
            dropped += fidx[face_matrix.shape[0]:]
            fobj = fobj[:face_matrix.shape[0], :]
    if not np.array_equal(face_matrix, fobj):
        return None
    return list(np.delete(np.asarray(obj_lines), dropped))

def sample_colored_cloud_epfl(obj_path, out_ply, n, n_points=N_POINTS,
                              occlusion_threshold=OCCLUSION_THRESHOLD, cc_bin=CC_BIN):
    """EPFL clean-mesh colored sampling: dedup interior faces, then CloudCompare.
    Returns out_ply. Raises on unrecoverable mesh mismatch (caught by the batch)."""
    in_dir, in_file = os.path.split(obj_path)
    fm_file, fq_file = "face_matrix.npy", "face_quality.npy"

    # 1) ambient occlusion in a separate process (EPFL does this for stability)
    subprocess.run(["python", _AO_HELPER, in_dir, in_file, fm_file, fq_file],
                   capture_output=True, timeout=600)
    fm_path, fq_path = os.path.join(in_dir, fm_file), os.path.join(in_dir, fq_file)
    if not (os.path.exists(fm_path) and os.path.exists(fq_path)):
        raise RuntimeError("ambient_occlusion produced no output (pymeshlab failed).")
    face_matrix = np.load(fm_path); face_quality = np.load(fq_path)
    os.remove(fm_path); os.remove(fq_path)

    # 2) which duplicate/interior faces to drop
    drop = _duplicate_faces_to_delete(face_matrix, face_quality, occlusion_threshold)

    # 3) rewrite a clean .obj with those faces removed
    with open(obj_path, "r") as f:
        obj_lines = f.read().splitlines()
    obj_lines = _ensure_face_correspondence(obj_lines, face_matrix)
    if obj_lines is None:
        raise RuntimeError("face mismatch between pymeshlab and .obj; skipping model.")
    nonempty = [(i, x) for i, x in enumerate(obj_lines) if len(x) > 0]
    fidx = np.asarray([i for i, x in nonempty if x[0] == 'f'])
    obj_lines = list(np.delete(np.asarray(obj_lines), fidx[drop]))

    clean_dir = os.path.join(EPFL_DIR, f"_clean_{n}", "models")
    os.makedirs(clean_dir, exist_ok=True)

    # copy the .mtl referenced in the obj, and point the obj at the local copy
    mtl_i, mtl_line = next((i, x) for i, x in enumerate(obj_lines) if "mtllib" in x)
    mtl_file = next(t for t in mtl_line.split() if t.endswith(".mtl"))
    clean_mtl = f"clean_{n}.mtl"
    shutil.copyfile(os.path.join(in_dir, mtl_file), os.path.join(clean_dir, clean_mtl))
    obj_lines[mtl_i] = f"mtllib {clean_mtl}"

    clean_obj = os.path.join(clean_dir, f"clean_{n}.obj")
    with open(clean_obj, "w") as f:
        f.write("\n".join(obj_lines))

    # copy every texture the .mtl references (so CloudCompare can read color)
    with open(os.path.join(clean_dir, clean_mtl)) as f:
        for line in f.read().splitlines():
            if "map_Kd" in line:
                tex = line.split()[1]
                rel, name = os.path.split(tex)
                dst_dir = os.path.join(clean_dir, rel); os.makedirs(dst_dir, exist_ok=True)
                shutil.copyfile(os.path.join(in_dir, tex), os.path.join(dst_dir, name))

    # 4) sample the CLEAN mesh with CloudCompare (same command as the speckle path)
    cmd = (f'xvfb-run -a {cc_bin} -SILENT -AUTO_SAVE OFF -C_EXPORT_FMT PLY '
           f'-O "{clean_obj}" -SAMPLE_MESH POINTS {n_points} -SAVE_CLOUDS')
    subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300)
    produced = [p for p in glob.glob(os.path.join(clean_dir, "*.ply"))
                if "SAMPLED_POINTS" in os.path.basename(p)]
    if not produced:
        raise RuntimeError("CloudCompare produced no PLY from the clean mesh.")
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    shutil.move(produced[0], out_ply)
    shutil.rmtree(os.path.join(EPFL_DIR, f"_clean_{n}"), ignore_errors=True)  # tidy up
    return out_ply


In [ ]:
# --- EPFL smoke test on the SAME single model used in Cell F (ts / tid / troot) ---
# Compare this clean output against the speckle output (tout) from Cell F.
_epfl_out = sample_colored_cloud_epfl(tobj, f"{EPFL_DIR}/{ts}/{tid}.ply", n=0)
_pc = o3d.io.read_point_cloud(_epfl_out)
_cols = np.asarray(_pc.colors)
print(f"EPFL clean -> {_epfl_out}  ({os.path.getsize(_epfl_out)/1e6:.1f} MB)")
print(f"points={len(_pc.points)}  has_colors={_pc.has_colors()}  color_std={_cols.std():.4f}")
print("Compare with the speckle PLY from Cell F to see the dual-face fix.")


In [ ]:
# ============================================================
# Cell G — Sample ALL selected models, robustly (RAW dense clouds only)
# ------------------------------------------------------------
# Per-model try/except so one bad mesh can't abort the batch; failures -> errors list.
# We write ONLY the raw 65,536-point colored PLY straight from CloudCompare -- NO FPS,
# NO normalization (those are deferred to a later derivation step). We record a manifest
# row per success, incl. color_ok (the silent-failure guard: a valid-but-colorless PLY
# means the texture chain broke or the model is textureless).
# ============================================================
import os, traceback
import numpy as np, open3d as o3d
from tqdm import tqdm

NAME_BY_SYNSET = {v: k for k, v in CATEGORIES.items()}

def is_color_ok(ply_path, std_thresh=0.01):
    pc = o3d.io.read_point_cloud(ply_path)
    return bool(pc.has_colors() and np.asarray(pc.colors).std() > std_thresh)

manifest_rows, errors = [], []
epfl_errors = []
epfl_idx = 0   # unique id per clean-mesh temp dir
jobs = [(s, mid, root) for s in SYNSETS for (mid, root) in selected[s]]

for synset, mid, root in tqdm(jobs, desc=f"Sampling {len(jobs)} models"):
    obj = os.path.join(root, "models", "model_normalized.obj")
    out = f"{PLY_DIR}/{synset}/{mid}.ply"
    epfl_out = f"{EPFL_DIR}/{synset}/{mid}.ply"
    try:
        sample_colored_cloud(obj, out)
        pc = o3d.io.read_point_cloud(out)
        row = {
            "synset": synset, "category": NAME_BY_SYNSET[synset], "model_id": mid,
            "source_obj": obj, "output_ply": out,
            "n_points": N_POINTS,
            "actual_points": len(pc.points),
            "color_ok": is_color_ok(out),
        }
        # EPFL clean-mesh path (separate output; failures here don't lose the speckle PLY)
        try:
            sample_colored_cloud_epfl(obj, epfl_out, n=epfl_idx)
            epfl_idx += 1
            row["epfl_ply"] = epfl_out
            row["epfl_color_ok"] = is_color_ok(epfl_out)
        except Exception:
            row["epfl_ply"] = ""
            row["epfl_color_ok"] = False
            epfl_errors.append((synset, mid, traceback.format_exc()))
        manifest_rows.append(row)
    except Exception:
        errors.append((synset, mid, traceback.format_exc()))

n_ok = sum(r["color_ok"] for r in manifest_rows)
print(f"\nSampled {len(manifest_rows)}/{len(jobs)} models; {len(errors)} failed.")
print(f"color_ok (speckle): {n_ok}/{len(manifest_rows)} "
      f"(colorless ones are likely textureless models, not bugs).")
n_epfl = sum(r.get("epfl_color_ok", False) for r in manifest_rows)
print(f"EPFL clean produced: {n_epfl}/{len(manifest_rows)}  "
      f"({len(epfl_errors)} EPFL-step failures -> speckle PLY still kept).")


In [ ]:
# ============================================================
# Cell H — Validate: see colors AND the dual-face speckle (Plotly, inline)
# ------------------------------------------------------------
# color_ok already flagged silent failures across all 60. Here we eyeball a few:
# Open3D's viewer can't run in Colab (no display), so we use a Plotly 3D scatter,
# downsampled for browser speed. Per ADR-0001, visible salt-and-pepper speckle is
# SUCCESS today (it is the problem we fix in a later iteration), not a failure.
# ============================================================
import numpy as np, open3d as o3d
import plotly.graph_objects as go

def show_cloud(ply_path, title, max_pts=25000):
    pc = o3d.io.read_point_cloud(ply_path)
    pts, cols = np.asarray(pc.points), np.asarray(pc.colors)
    if len(pts) > max_pts:                       # keep the browser smooth
        idx = np.random.choice(len(pts), max_pts, replace=False)
        pts, cols = pts[idx], (cols[idx] if len(cols) else cols)
    marker = dict(size=1.5)
    if len(cols):
        marker["color"] = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in cols]
    fig = go.Figure(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                                 mode="markers", marker=marker))
    fig.update_layout(title=title, height=500, scene_aspectmode="data",
                      margin=dict(l=0, r=0, t=30, b=0))
    fig.show()

# one example per category (first successful model)
for synset in SYNSETS:
    row = next((r for r in manifest_rows if r["synset"] == synset), None)
    if row:
        show_cloud(row["output_ply"],
                   f'{row["category"]} — {row["model_id"]} (color_ok={row["color_ok"]})')
    else:
        print(f"No successful model to show for {synset}.")


In [ ]:
# ============================================================
# Cell I — Hand off to the shared Ortak folder
# ------------------------------------------------------------
# Write manifest.csv + errors.log locally, then copy the whole colored_pc/ tree
# (60 small PLYs) from local disk to Ortak/colored_pc/ in ONE batch. This shared
# write IS the transfer: both teammates then see identical files (ADR-0002 / 0003).
# ============================================================
import os, shutil
import pandas as pd

pd.DataFrame(manifest_rows).to_csv(f"{PLY_DIR}/manifest.csv", index=False)
with open(f"{PLY_DIR}/errors.log", "w") as fh:
    if not errors:
        fh.write("No errors.\n")
    for synset, mid, tb in errors:
        fh.write(f"\n=== {synset}/{mid} ===\n{tb}\n")

# Local -> shared Drive (dirs_exist_ok so re-runs overwrite cleanly).
shutil.copytree(PLY_DIR, OUT_DIR, dirs_exist_ok=True)
print("Handoff complete ->", OUT_DIR)
for root, dirs, files in os.walk(OUT_DIR):
    lvl = root.replace(OUT_DIR, "").count(os.sep)
    print("   " * lvl, os.path.basename(root.rstrip("/")) + "/")
    for f in sorted(files)[:6]:
        print("   " * (lvl + 1), f)


# Post-processing: dense 2^20 EPFL clouds -> target-size GT

Everything above PRODUCES the raw, dense (2^20-pt) colored clouds. This SEPARATE step
takes the dense **EPFL** clouds and downsamples each to `N_GT_TARGET`, carrying color.

**Method (geometric base + color-edge budget):**
1. Geometric FPS lays down most of the budget, spread evenly over the surface
   (geometry-aware; this base is guaranteed and never distorted).
2. If `COLOR_FRAC>0`, the remaining budget is drawn toward color edges (logos, stripes,
   block boundaries) weighted by color gradient -- extra points where color changes,
   WITHOUT stealing from the geometric base.

Dense PLYs stay untouched as the SOURCE -- re-run with a different target / `COLOR_FRAC`
without re-sampling. **Start with `COLOR_FRAC=0.0` (plain FPS); raise only if the star /
text / edges blur at the target size.**

In [ ]:
# ============================================================
# Cell P1 - Post-processing functions: geometric FPS + color-edge budget
# ============================================================
import os, glob
import numpy as np, open3d as o3d
from scipy.spatial import cKDTree

def farthest_point_sampling(points, n_samples, seed=None):
    """Geometric FPS on xyz only; color rides along via returned indices.
    GPU/torch when available (~50-150x faster than numpy on ~1M pts), with a
    numpy CPU fallback. Start point uses np.random.default_rng(seed) on both
    paths, so results are seed-reproducible and match the old numpy path."""
    N = points.shape[0]
    if n_samples >= N:
        return np.arange(N)
    xyz = np.ascontiguousarray(points[:, :3], dtype=np.float32)
    start = int(np.random.default_rng(seed).integers(N))
    try:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("no cuda")
        dev = torch.device("cuda")
        pts = torch.as_tensor(xyz, dtype=torch.float32, device=dev)
        sel = torch.empty(n_samples, dtype=torch.int64, device=dev)
        dist = torch.full((N,), float("inf"), device=dev)
        sel[0] = start
        last = start
        for i in range(1, n_samples):
            d = ((pts - pts[last]) ** 2).sum(1)
            dist = torch.minimum(dist, d)
            last = int(torch.argmax(dist))
            sel[i] = last
        return sel.cpu().numpy()
    except Exception:
        sel = np.zeros(n_samples, dtype=np.int64)
        dist = np.full(N, np.inf)
        sel[0] = start
        for i in range(1, n_samples):
            d = np.sum((xyz - xyz[sel[i - 1]]) ** 2, axis=1)
            dist = np.minimum(dist, d)
            sel[i] = np.argmax(dist)
        return sel

def color_gradient(points, k=16):
    """Per-point color gradient: |rgb - mean(neighbor rgb)|. ~0 on flat regions,
    high near color edges. Pure float kNN, no grid/voxels."""
    xyz, rgb = points[:, :3], points[:, 3:6]
    tree = cKDTree(xyz)
    _, idx = tree.query(xyz, k=min(k + 1, len(xyz)))
    neigh = rgb[idx[:, 1:]]
    return np.linalg.norm(rgb - neigh.mean(axis=1), axis=1)

def color_aware_sample(points, n_total, color_frac=0.0, k=16, seed=42):
    """Downsample to n_total -> RAW float (n_total, 6) [xyz+rgb]. No voxelization.
      color_frac = 0.0 -> pure geometric FPS.
      color_frac > 0   -> geometric FPS lays the base (1-color_frac of the budget),
                          then the rest is drawn toward color edges (gradient-weighted).
                          Geometry base is NOT distorted; color only ADDS at edges.
    Flat-colored model (gradient ~ 0) -> color budget falls back to random; harmless."""
    rng = np.random.default_rng(seed)
    N = points.shape[0]
    if n_total >= N:
        return points.copy()
    n_color = int(n_total * color_frac)
    n_geo = n_total - n_color
    geo_idx = farthest_point_sampling(points, n_geo, seed=seed)
    if n_color == 0:
        return points[geo_idx]
    mask = np.ones(N, dtype=bool); mask[geo_idx] = False
    rem = np.nonzero(mask)[0]
    grad = color_gradient(points, k=k)[rem]
    if grad.sum() <= 0:
        col_idx = rng.choice(rem, size=min(n_color, len(rem)), replace=False)
    else:
        col_idx = rng.choice(rem, size=min(n_color, len(rem)), replace=False, p=grad / grad.sum())
    return points[np.concatenate([geo_idx, col_idx])]

def load_ply_xyzrgb(ply_path):
    pc = o3d.io.read_point_cloud(ply_path)
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if rgb.shape[0] != xyz.shape[0]:
        rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], axis=1).astype(np.float32)

def save_ply_xyzrgb(arr, out_ply):
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(arr[:, :3].astype(np.float64))
    pc.colors = o3d.utility.Vector3dVector(np.clip(arr[:, 3:6], 0, 1).astype(np.float64))
    os.makedirs(os.path.dirname(out_ply), exist_ok=True)
    o3d.io.write_point_cloud(out_ply, pc)


In [ ]:
# --- Post-processing smoke test on the single EPFL model (ts / tid) ---
_src = f"{EPFL_DIR}/{ts}/{tid}.ply"
_dense = load_ply_xyzrgb(_src)
print(f"dense source: {_dense.shape[0]} pts")
_gt = color_aware_sample(_dense, N_GT_TARGET, color_frac=COLOR_FRAC, k=KNN_COLOR, seed=GT_SEED)
_out = f"{GT_DIR}/{ts}/{tid}.ply"
save_ply_xyzrgb(_gt, _out)
print(f"GT: {_gt.shape} -> {_out}")
print(f"color_frac={COLOR_FRAC} (0 = plain FPS). Open this PLY and check the star / "
      f"'Liberty Jet' text / blue-white edge are still readable.")


In [ ]:
# ============================================================
# Cell P2 - Batch post-process ALL EPFL clouds -> GT
# ============================================================
import traceback, shutil
from tqdm import tqdm

gt_rows, gt_errors = [], []
epfl_plys = sorted(glob.glob(f"{EPFL_DIR}/*/*.ply"))
print(f"Found {len(epfl_plys)} EPFL clouds to downsample.")

for src in tqdm(epfl_plys, desc=f"Downsampling -> {N_GT_TARGET} pts"):
    synset = os.path.basename(os.path.dirname(src))
    mid = os.path.splitext(os.path.basename(src))[0]
    out = f"{GT_DIR}/{synset}/{mid}.ply"
    try:
        dense = load_ply_xyzrgb(src)
        gt = color_aware_sample(dense, N_GT_TARGET, color_frac=COLOR_FRAC, k=KNN_COLOR, seed=GT_SEED)
        save_ply_xyzrgb(gt, out)
        gt_rows.append({"synset": synset, "model_id": mid, "gt_ply": out,
                        "dense_points": dense.shape[0], "gt_points": gt.shape[0],
                        "color_frac": COLOR_FRAC})
    except Exception:
        gt_errors.append((synset, mid, traceback.format_exc()))

print(f"\nGT produced: {len(gt_rows)}/{len(epfl_plys)}  ({len(gt_errors)} failed).")
if gt_rows:
    os.makedirs(GT_OUT_DIR, exist_ok=True)
    shutil.copytree(GT_DIR, GT_OUT_DIR, dirs_exist_ok=True)
    print(f"Copied GT to {GT_OUT_DIR}")


---
# PART 2 — PCN-style partial generation (3 methods, for comparison)

Part 1 produced, per model: the **dense EPFL colored cloud** (local at `EPFL_DIR/<synset>/<id>.ply`) and the
extracted **mesh** (`selected[...]`). When iterating on partials, re-run **only the Part-2 cells**.

**PCN's recipe (§5.1):** the partial is made by rendering a 2.5D **depth image from the mesh** and
**back-projecting** it to 3D — 8 random viewpoints, *not* a subset of the complete cloud. PCN is
geometry-only; we are the **colored** study, so our partial also carries `rgb`.

We build **THREE** methods and ship all of them (see `docs/adr/0005`):

| method | geometry | color | tradeoff |
|---|---|---|---|
| `meshray_nn` *(reference)* | mesh raycast → back-project (**faithful PCN**) | nearest-neighbour from the cleaned dense cloud | color is sub-mm-exact + robust, but *transferred* not "direct" |
| `meshray_tex` | **same** raycast geometry | texture lookup at the exact ray-hit | literally-direct color, but brittle (UV/material) → **falls back to NN** |
| `ptproj` | project dense colored cloud + z-buffer | direct, from the surviving point | simplest + direct color, but **sees through gaps**; partial ⊂ dense |

**Clean comparison:** all three share the **same 8 seeded viewpoints** per model, so
`meshray_nn` vs `meshray_tex` is a pure **color** test (identical geometry) and `meshray_*` vs `ptproj` is a
pure **geometry** test. Output: `data_s4/{complete, partial/<method>}/...` + `manifest.csv` + `viewpoints.csv`.


In [ ]:
# ============================================================
# Cell Q0 — PART-2 config (partial-generation knobs)
# ------------------------------------------------------------
# Part 1 already set ORTAK / RUN_TAG / EPFL_DIR / WORK_DIR and produced the dense clouds + GT.
# These knobs control ONLY the partial generation. See ADR-0005 for why each default.
# ============================================================
N_VIEWS         = 8        # PCN uses 8 random viewpoints per model
VIEW_SEED       = 7        # fixes the 8 camera poses -> SAME poses for all 3 methods (fair compare)
RENDER_RES      = 512      # depth-image / projection resolution (W=H). ~1M dense pts @ 512^2 ~= 4 pts/px ->
                           # ptproj occludes well (verified). RAISING res thins pts/pixel and INCREASES ptproj
                           # see-through; keep RES^2 comfortably below the dense point count.
FOV_DEG         = 60       # camera vertical field-of-view (degrees)
CAM_DIST_FACTOR = 2.2      # camera distance = factor * object bounding radius
HEMISPHERE      = False    # True -> draw viewpoints from the upper hemisphere only (real sensors rarely see undersides)

METHODS         = ["meshray_nn", "meshray_tex", "ptproj"]   # the three partial methods (ADR-0005)

# Output layout (PCN-style: complete/ + partial/<method>/, 8 numbered partials per model).
# Write locally first, copy to the shared Ortak folder once at the end (same handoff pattern as s2/s3).
PARTIAL_LOCAL   = f"{WORK_DIR}/partial"
COMPLETE_LOCAL  = f"{WORK_DIR}/complete"
COMPLETE_OUT    = f"{ORTAK}/data_{RUN_TAG}/complete"
PARTIAL_OUT     = {m: f"{ORTAK}/data_{RUN_TAG}/partial/{m}" for m in METHODS}

print("Part-2 config:", N_VIEWS, "views @", f"{RENDER_RES}px,", "methods:", METHODS)
print("GT size (from Part 1):", N_GT_TARGET, "pts  | writing pairs to:", f"{ORTAK}/data_{RUN_TAG}")


In [ ]:
# ============================================================
# Cell Q1 — Camera + viewpoint utilities  (SHARED by all 3 methods)
# ------------------------------------------------------------
# One pinhole camera, used two ways: the meshray_* methods shoot rays OUT through pixels into the mesh;
# ptproj projects dense points IN to pixels. Both use the SAME look-at + intrinsics, so a difference
# between methods is the METHOD, never the camera. (This math was unit-tested: project<->ray round-trips
# to 1e-17 and the z-buffer correctly occludes a back plane.)
# Convention: camera frame z=forward(look dir), x=right, y=down (OpenCV-style). R = cam->world (columns
# are the camera axes in world coords). Image mirroring is irrelevant for point clouds.
# ============================================================
import numpy as np

def make_viewpoints(center, radius, n, seed, hemi=False):
    """n camera positions on a sphere of `radius` around `center` (seeded -> reproducible & shared)."""
    rng = np.random.default_rng(seed)
    d = rng.normal(size=(n, 3))                       # gaussian -> uniform direction once normalized
    if hemi:
        d[:, 1] = np.abs(d[:, 1])                     # keep the +y (upper) hemisphere
    d /= np.linalg.norm(d, axis=1, keepdims=True)
    return center + d * radius

def look_at_R(cam_pos, target, world_up=np.array([0., 1., 0.])):
    """Rotation cam->world for a camera at cam_pos looking at target."""
    f = target - cam_pos; f = f / np.linalg.norm(f)          # forward = z_cam
    if abs(f @ world_up) > 0.999:                            # forward ~ up -> pick a different up
        world_up = np.array([0., 0., 1.])
    r = np.cross(f, world_up); r = r / np.linalg.norm(r)     # right = x_cam
    dn = np.cross(f, r)                                       # down  = y_cam
    return np.stack([r, dn, f], axis=1)                       # columns = cam axes in world

def intrinsics(W, H, fov_deg):
    """Pinhole fx,fy,cx,cy for a W x H image with vertical FOV fov_deg (square pixels)."""
    fy = (H / 2.) / np.tan(np.radians(fov_deg) / 2.)
    return fy, fy, W / 2., H / 2.

def build_ray_dirs(R, fx, fy, cx, cy, W, H):
    """World-space ray direction per pixel (origin = cam_pos). Shape (H,W,3)."""
    uu, vv = np.meshgrid(np.arange(W) + 0.5, np.arange(H) + 0.5)
    dc = np.stack([(uu - cx) / fx, (vv - cy) / fy, np.ones_like(uu)], -1)   # +z forward in cam frame
    dc /= np.linalg.norm(dc, axis=-1, keepdims=True)
    return dc @ R.T                                                          # rotate cam->world

def project_points(P, cam_pos, R, fx, fy, cx, cy, W, H):
    """Project world points P (N,3) into the image. Returns u,v (pixels), zc (depth), valid mask."""
    pc = (P - cam_pos) @ R                                                   # world->cam (= R.T @ (P-cam))
    zc = pc[:, 2]
    safe = np.where(zc == 0, 1e-9, zc)
    u = fx * pc[:, 0] / safe + cx
    v = fy * pc[:, 1] / safe + cy
    valid = (zc > 1e-9) & (u >= 0) & (u < W) & (v >= 0) & (v < H)            # in front & inside the image
    return u, v, zc, valid

def zbuffer_keep(u, v, zc, valid, W):
    """Keep the NEAREST point per pixel (the z-buffer). Returns indices into P that survive."""
    idx = np.nonzero(valid)[0]
    pix = np.floor(v[valid]).astype(int) * W + np.floor(u[valid]).astype(int)
    order = np.argsort(zc[valid])                                            # nearest first
    pix_s, idx_s = pix[order], idx[order]
    _, first = np.unique(pix_s, return_index=True)                           # first (nearest) per pixel
    return idx_s[first]

def object_center_radius(dense_xyz):
    c = (dense_xyz.min(0) + dense_xyz.max(0)) / 2.0
    r = float(np.linalg.norm(dense_xyz - c, axis=1).max())
    return c, r

def cameras_for_model(dense_xyz):
    """The 8 shared cameras for one model, sized to its bounding sphere."""
    c, r = object_center_radius(dense_xyz)
    vps = make_viewpoints(c, CAM_DIST_FACTOR * r, N_VIEWS, VIEW_SEED, HEMISPHERE)
    R_list = [look_at_R(vp, c) for vp in vps]
    intr = intrinsics(RENDER_RES, RENDER_RES, FOV_DEG)
    return vps, R_list, intr, c, r

print("Camera utilities ready (shared by meshray_* and ptproj).")


In [ ]:
# ============================================================
# Cell Q2 — The THREE partial methods
# ------------------------------------------------------------
# Each method card states WHAT it makes, HOW, and the TRADEOFF. All consume the same per-model cameras
# from Cell Q1, so they are directly comparable.
# ============================================================
import os
import numpy as np, open3d as o3d, open3d.core as o3c
from scipy.spatial import cKDTree

# --- mesh raycaster (geometry shared by meshray_nn and meshray_tex) ---
def load_mesh_for_raycast(obj_path):
    """Tensor mesh keeps per-triangle texture_uvs aligned to triangle ids (needed by meshray_tex)."""
    tmesh = o3d.t.io.read_triangle_mesh(obj_path)
    scene = o3d.t.geometry.RaycastingScene()
    scene.add_triangles(tmesh)
    return scene, tmesh

def raycast_view(scene, cam_pos, R, fx, fy, cx, cy, W, H):
    """FAITHFUL PCN geometry: one ray per pixel into the mesh; back-project the hits to world xyz.
    Solid triangles occlude exactly -> true single-view surface, perspective density, variable size.
    Returns (pts world (M,3), prim_ids (M,), prim_uvs (M,2)) for the M hit pixels."""
    dirs = build_ray_dirs(R, fx, fy, cx, cy, W, H).reshape(-1, 3).astype(np.float32)
    origins = np.broadcast_to(cam_pos.astype(np.float32), dirs.shape)
    rays = o3c.Tensor(np.concatenate([origins, dirs], 1), dtype=o3c.Dtype.Float32)
    ans = scene.cast_rays(rays)
    t = ans["t_hit"].numpy()
    hit = np.isfinite(t)
    pts = origins[hit] + t[hit, None] * dirs[hit]
    return pts, ans["primitive_ids"].numpy()[hit], ans["primitive_uvs"].numpy()[hit]

# --- color option A1: nearest-neighbour from the cleaned dense cloud ---
def color_by_nn(pts, dense_xyz, dense_rgb, _tree=None):
    """meshray_nn color. WHAT: rgb for each back-projected point. HOW: copy from the nearest dense point.
    TRADEOFF: 'transferred' not direct -- but at ~1M dense pts the neighbour is sub-mm, so it equals the
    texture color, already cleaned of dual-face speckle (see ADR-0005 / CONTEXT 'dual-face problem')."""
    tree = _tree or cKDTree(dense_xyz)
    _, idx = tree.query(pts, k=1)
    return dense_rgb[idx]

# --- color option A2: direct texture lookup at the ray hit ---
def load_texture(obj_path):
    """Best-effort: first map_Kd in the .mtl beside the OBJ -> HxWx3 float[0,1]. None if unavailable."""
    mtl = obj_path[:-4] + ".mtl"
    if not os.path.exists(mtl):
        return None
    base = os.path.dirname(obj_path)
    img = None
    for line in open(mtl, errors="ignore"):
        s = line.strip()
        if s.lower().startswith("map_kd"):
            img = os.path.join(base, s.split(maxsplit=1)[1].strip())       # may contain ../images/...
            break
    if not img or not os.path.exists(img):
        return None
    from PIL import Image
    return np.asarray(Image.open(img).convert("RGB"), dtype=np.float32) / 255.0

def color_by_texture(prim_ids, prim_uvs, tmesh, tex_img):
    """meshray_tex color. WHAT: rgb from the actual texture at the exact hit. HOW: barycentric-interpolate
    the hit triangle's UVs, sample the texture image. TRADEOFF: literally-direct, but UV/material handling
    is brittle and re-opens the dual-face color risk -> caller falls back to NN on any failure."""
    tri_uv = tmesh.triangle["texture_uvs"].numpy()                          # (n_tri,3,2) or KeyError
    uvt = tri_uv[prim_ids]                                                   # (M,3,2)
    w1, w2 = prim_uvs[:, 0], prim_uvs[:, 1]; w0 = 1.0 - w1 - w2             # barycentric weights
    uv = w0[:, None] * uvt[:, 0] + w1[:, None] * uvt[:, 1] + w2[:, None] * uvt[:, 2]
    th, tw = tex_img.shape[:2]
    tx = np.clip((uv[:, 0] % 1.0) * (tw - 1), 0, tw - 1).astype(int)
    ty = np.clip((1.0 - uv[:, 1] % 1.0) * (th - 1), 0, th - 1).astype(int)  # OBJ uv origin = bottom-left
    return tex_img[ty, tx]

# --- method B1: project the dense colored cloud through the SAME camera + z-buffer ---
def ptproj_view(dense_xyz, dense_rgb, cam_pos, R, fx, fy, cx, cy, W, H):
    """ptproj (teammate's idea, made rigorous). WHAT: a colored partial. HOW: project every dense point
    into the image, keep the nearest per pixel (z-buffer); color rides along on the surviving point.
    TRADEOFF: color is DIRECT, but a point cloud isn't solid -> it can SEE THROUGH gaps to the back
    surface (worst on thin parts), and the partial is a SUBSET of the dense cloud (can share GT points)."""
    u, v, zc, valid = project_points(dense_xyz, cam_pos, R, fx, fy, cx, cy, W, H)
    keep = zbuffer_keep(u, v, zc, valid, W)
    return dense_xyz[keep], dense_rgb[keep]

# --- unified driver: all requested methods, all views, for one model ---
def make_partials(obj_path, dense_xyz, dense_rgb, viewpoints, R_list, intr, W, H, methods):
    fx, fy, cx, cy = intr
    out = {m: [] for m in methods}
    info = {m: {"ok": 0, "fallback": 0} for m in methods}
    need_ray = any(m.startswith("meshray") for m in methods)
    tree = cKDTree(dense_xyz) if "meshray_nn" in methods or "meshray_tex" in methods else None
    scene = tmesh = tex = None
    if need_ray:
        scene, tmesh = load_mesh_for_raycast(obj_path)
        if "meshray_tex" in methods:
            try: tex = load_texture(obj_path)
            except Exception: tex = None
    for vp, R in zip(viewpoints, R_list):
        if need_ray:
            pts, pid, puv = raycast_view(scene, vp, R, fx, fy, cx, cy, W, H)
        if "meshray_nn" in methods:
            out["meshray_nn"].append((pts, color_by_nn(pts, dense_xyz, dense_rgb, tree)))
            info["meshray_nn"]["ok"] += 1
        if "meshray_tex" in methods:
            try:
                if tex is None: raise ValueError("no texture/UVs")
                col = color_by_texture(pid, puv, tmesh, tex); info["meshray_tex"]["ok"] += 1
            except Exception:
                col = color_by_nn(pts, dense_xyz, dense_rgb, tree); info["meshray_tex"]["fallback"] += 1
            out["meshray_tex"].append((pts, col))
        if "ptproj" in methods:
            out["ptproj"].append(ptproj_view(dense_xyz, dense_rgb, vp, R, fx, fy, cx, cy, W, H))
            info["ptproj"]["ok"] += 1
    return out, info

print("Three partial methods ready:", METHODS)


In [ ]:
# ============================================================
# Cell Q3 — SMOKE TEST on ONE model (all 3 methods + GT, visualized)  <-- run this before the batch
# ------------------------------------------------------------
# Exercises the whole Part-2 path on the single Part-1 smoke model (ts / tid / troot). Eyeball: each
# partial should look like a ONE-SIDED view; meshray_* colors should match the GT; ptproj may show
# see-through on thin parts. This is the gate before the (longer) batch.
# ============================================================
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def show_clouds(clouds, max_pts=20000, marker=1.6):
    """clouds: list of (xyz, rgb, title) -> a row of 3D scatter subplots."""
    n = len(clouds)
    fig = make_subplots(rows=1, cols=n, specs=[[{"type": "scene"} for _ in range(n)]],
                        subplot_titles=[t for *_, t in clouds])
    for i, (xyz, rgb, _t) in enumerate(clouds, 1):
        if len(xyz) > max_pts:
            s = np.random.choice(len(xyz), max_pts, replace=False); xyz, rgb = xyz[s], rgb[s]
        col = ([f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in rgb]
               if len(rgb) else None)
        fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                   marker=dict(size=marker, color=col)), row=1, col=i)
    fig.update_layout(height=430, margin=dict(l=0, r=0, t=30, b=0), showlegend=False)
    fig.update_scenes(aspectmode="data")
    fig.show()

_dense = load_ply_xyzrgb(f"{EPFL_DIR}/{ts}/{tid}.ply")          # dense colored cloud (Part 1)
_dx, _drgb = _dense[:, :3], _dense[:, 3:6]
_obj = os.path.join(troot, "models", "model_normalized.obj")   # the mesh (Part 1)
_vps, _Rl, _intr, _c, _r = cameras_for_model(_dx)

# one view is enough for the smoke test
_parts, _info = make_partials(_obj, _dx, _drgb, _vps[:1], _Rl[:1], _intr, RENDER_RES, RENDER_RES, METHODS)
_gt = color_aware_sample(_dense, N_GT_TARGET, color_frac=COLOR_FRAC, k=KNN_COLOR, seed=GT_SEED)

print(f"model {ts}/{tid}  dense={len(_dx)}  GT={len(_gt)}")
for m in METHODS:
    xyz, rgb = _parts[m][0]
    print(f"  {m:12s}: {len(xyz):6d} pts  color_std={rgb.std():.3f}  info={_info[m]}")

show_clouds([(_parts[m][0][0], _parts[m][0][1], m) for m in METHODS]
            + [(_gt[:, :3], _gt[:, 3:6], f"GT ({len(_gt)})")])


In [ ]:
# ============================================================
# Cell Q4 — BATCH: every model x 8 views x 3 methods + GT  ->  pairs + manifest
# ------------------------------------------------------------
# Per-model try/except so one bad mesh can't abort the batch. We write the GT once per model and
# 8 partials per (method, model). The manifest pairs each partial -> its GT and records the
# color_ok / fallback flags; viewpoints.csv stores the shared camera poses for reproducibility.
# ============================================================
import os, traceback
import numpy as np
from tqdm import tqdm

rows, vp_rows, errs = [], [], []
jobs = [(s, mid, root) for s in SYNSETS for (mid, root) in selected[s]]
print(f"Generating pairs for {len(jobs)} models x {N_VIEWS} views x {len(METHODS)} methods...")

for synset, mid, root in tqdm(jobs, desc="pairs"):
    try:
        dense = load_ply_xyzrgb(f"{EPFL_DIR}/{synset}/{mid}.ply")
        dx, drgb = dense[:, :3], dense[:, 3:6]
        obj = os.path.join(root, "models", "model_normalized.obj")
        vps, Rl, intr, c, r = cameras_for_model(dx)

        # complete GT (once per model) — FPS of the dense cloud (s3's function)
        gt = color_aware_sample(dense, N_GT_TARGET, color_frac=COLOR_FRAC, k=KNN_COLOR, seed=GT_SEED)
        gt_out = f"{COMPLETE_LOCAL}/{synset}/{mid}.ply"
        save_ply_xyzrgb(gt, gt_out)

        # partials — all methods, all views
        parts, info = make_partials(obj, dx, drgb, vps, Rl, intr, RENDER_RES, RENDER_RES, METHODS)
        for m in METHODS:
            for vi, (xyz, rgb) in enumerate(parts[m]):
                p_out = f"{PARTIAL_LOCAL}/{m}/{synset}/{mid}/{vi:02d}.ply"
                save_ply_xyzrgb(np.concatenate([xyz, rgb], 1), p_out)
                rows.append(dict(synset=synset, category=NAME_BY_SYNSET[synset], model_id=mid,
                                 method=m, view=vi, n_points=len(xyz),
                                 color_ok=bool(rgb.std() > 0.01),
                                 partial_ply=p_out, gt_ply=gt_out, gt_points=len(gt)))
        for vi, vp in enumerate(vps):
            vp_rows.append(dict(synset=synset, model_id=mid, view=vi,
                                cam_x=vp[0], cam_y=vp[1], cam_z=vp[2],
                                ctr_x=c[0], ctr_y=c[1], ctr_z=c[2], radius=r))
    except Exception:
        errs.append((synset, mid, traceback.format_exc()))

tex_fb = sum(1 for x in rows if x["method"] == "meshray_tex" and not x["color_ok"])
print(f"\nDone. {len(rows)} partial files for {len(jobs) - len(errs)}/{len(jobs)} models "
      f"({len(errs)} failed). meshray_tex colorless/fallback files: {tex_fb}.")


In [ ]:
# ============================================================
# Cell Q5 — VALIDATE: the 3 methods side by side on a couple of models (same view)
# ------------------------------------------------------------
# This is the comparison the supervisor/teammate looks at: identical geometry between meshray_nn and
# meshray_tex (color-only difference), and meshray_* vs ptproj (geometry difference / see-through).
# ============================================================
import numpy as np, open3d as o3d

def _load(p):
    pc = o3d.io.read_point_cloud(p)
    return np.asarray(pc.points), np.asarray(pc.colors)

for synset in SYNSETS:
    row = next((x for x in rows if x["synset"] == synset and x["view"] == 0
                and x["method"] == "meshray_nn"), None)
    if not row:
        print(f"(no successful model to show for {synset})"); continue
    mid = row["model_id"]
    panels = []
    for m in METHODS:
        xyz, rgb = _load(f"{PARTIAL_LOCAL}/{m}/{synset}/{mid}/00.ply")
        panels.append((xyz, rgb, f"{m} ({len(xyz)})"))
    gxyz, grgb = _load(f"{COMPLETE_LOCAL}/{synset}/{mid}.ply")
    panels.append((gxyz, grgb, f"GT ({len(gxyz)})"))
    print(f"{NAME_BY_SYNSET[synset]} — {mid}  (view 0)")
    show_clouds(panels)


In [ ]:
# ============================================================
# Cell Q6 — HANDOFF: write manifest + viewpoints, copy local -> shared Ortak/data_s4
# ------------------------------------------------------------
# The shared write IS the transfer (ADR-0002/0003): both teammates then see identical pairs.
# Layout:  data_s4/complete/<synset>/<id>.ply  and  data_s4/partial/<method>/<synset>/<id>/00..07.ply
# ============================================================
import os, shutil
import pandas as pd

os.makedirs(PARTIAL_LOCAL, exist_ok=True)
pd.DataFrame(rows).to_csv(f"{PARTIAL_LOCAL}/manifest.csv", index=False)
pd.DataFrame(vp_rows).to_csv(f"{PARTIAL_LOCAL}/viewpoints.csv", index=False)
with open(f"{PARTIAL_LOCAL}/errors.log", "w") as fh:
    fh.write("No errors.\n" if not errs else "")
    for s, m, tb in errs:
        fh.write(f"\n=== {s}/{m} ===\n{tb}\n")

base = f"{ORTAK}/data_{RUN_TAG}"
os.makedirs(base, exist_ok=True)
shutil.copytree(COMPLETE_LOCAL, COMPLETE_OUT, dirs_exist_ok=True)
for m in METHODS:
    shutil.copytree(f"{PARTIAL_LOCAL}/{m}", PARTIAL_OUT[m], dirs_exist_ok=True)
shutil.copy(f"{PARTIAL_LOCAL}/manifest.csv", f"{base}/manifest.csv")
shutil.copy(f"{PARTIAL_LOCAL}/viewpoints.csv", f"{base}/viewpoints.csv")
shutil.copy(f"{PARTIAL_LOCAL}/errors.log", f"{base}/errors.log")

print("Handoff complete ->", base)
print("  complete/  + partial/{%s}/  + manifest.csv + viewpoints.csv" % ", ".join(METHODS))


## PART 3 — Resilient HF-native driver (ADR-0008)

**Run this instead of Cells G / P2 / Q4.** First run the function cells (A, B, C, E, F-EPFL, P1, Q0, Q1, Q2), then the cell below. It builds one model at a time → writes `{model_id}.npz` → streams to Hugging Face. A Colab disconnect costs one model; re-run the cell to resume.

In [ ]:
# ============================================================
# PART 3 — Resilient HF-native driver (ADR-0008)  [replaces Cells G / P2 / Q4]
# ------------------------------------------------------------
# Per-model atomic loop: skip-if-already-on-HF -> build (dense EPFL -> GT + meshray_nn
# partials) -> {model_id}.npz (atomic) -> CommitScheduler streams it to HF. A Colab
# disconnect costs ONE model; recovery is just re-running THIS cell.
# Prereqs: run cells A, B, C, E, F-EPFL, P1, Q0, Q1, Q2 first (they define the functions).
# ============================================================
import os, sys, subprocess

REPO_DIR = "/content/point-cloud-completion-research"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/efeyenice/point-cloud-completion-research.git", REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
from tools.pc_resilient import resilient_datagen, start_committer, hf_done_ids, anti_idle

def _secret(name):
    """Read from Colab Secrets (UI runs) OR os.environ (headless colab exec). Never hardcode."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

HF_TOKEN, REPO = _secret("HF_TOKEN"), "efeyenice/pc-completion-data"
LOCAL, SCRATCH = "/content/npz", "/content/scratch"

def _cameras(dx):
    vps, Rl, intr, *_ = cameras_for_model(dx)
    return vps, Rl, intr

def _gt(dense):
    return color_aware_sample(dense, N_GT_TARGET, color_frac=COLOR_FRAC, k=KNN_COLOR, seed=GT_SEED)

def _mkp(o, dx, drgb, vps, Rl, intr):
    parts, _ = make_partials(o, dx, drgb, vps, Rl, intr, RENDER_RES, RENDER_RES, ["meshray_nn"])
    return parts

fns = dict(epfl_sample=lambda o, out, n: sample_colored_cloud_epfl(o, out, n=n),
           load_ply=load_ply_xyzrgb, gt_sample=_gt, cameras=_cameras, make_partials=_mkp)
jobs = [(s, NAME_BY_SYNSET[s], mid, os.path.join(root, "models", "model_normalized.obj"))
        for s in SYNSETS for (mid, root) in selected[s]]

anti_idle()
sched = start_committer(LOCAL, REPO, HF_TOKEN, every_min=5)
resilient_datagen(jobs, fns, dict(local_dir=LOCAL, scratch_dir=SCRATCH, method="meshray_nn"),
                  done=hf_done_ids(REPO, HF_TOKEN))
sched.trigger()   # final flush to HF
